# 05 · Seleção manual e sync Kaggle Dataset → SSD

Escolha somente os arquivos que quer carregar. O sync não baixa o Dataset inteiro: ele usa o filtro `-f` do Kaggle CLI para transferir apenas os arquivos selecionados.


In [ ]:
import sys
import os
from pathlib import Path

SCRIPTS_DIR = Path("/kaggle/working/scripts")
sys.path.insert(0, str(SCRIPTS_DIR))

def _get_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def resolve_dataset_name(override=None):
    """Resolve o dataset Kaggle: override explícito ou KAGGLE_USERNAME/KAGGLE_DATASET_NAME."""
    if override:
        return override
    username = _get_secret("KAGGLE_USERNAME")
    dataset_name = _get_secret("KAGGLE_DATASET_NAME")
    if not username or not dataset_name:
        raise ValueError(
            "Dataset Kaggle não resolvido. Configure os Secrets do Kaggle:\n"
            "  - KAGGLE_USERNAME: seu username Kaggle\n"
            "  - KAGGLE_DATASET_NAME: nome do dataset (ex: comfydocs)\n"
            "Ou defina DATASET_OVERRIDE no início desta célula."
        )
    return f"{username}/{dataset_name}"

DATASET_OVERRIDE = None  # ex: "meuusuario/meudataset" para forçar
DATASET = resolve_dataset_name(DATASET_OVERRIDE)
print(f"[INFO] Dataset alvo: {DATASET}")

TARGET_DIR = Path("/kaggle/working/ComfyUI/models")

from kaggle_sync import select_dataset_files, sync_dataset_to_local

AUTO_SELECT_SINGLE = True
SELECTED_FILES = select_dataset_files(
    dataset=DATASET,
    auto_select_single=AUTO_SELECT_SINGLE,
)
print(f"[INFO] Seleção confirmada: {len(SELECTED_FILES)} arquivo(s)")

In [ ]:
# A seleção já foi confirmada pela célula anterior via input().
print(f"[INFO] Paths selecionados: {SELECTED_FILES}")

In [ ]:
# Download seletivo: somente os paths confirmados são enviados ao sync.
stats = sync_dataset_to_local(
    dataset=DATASET,
    target_dir=TARGET_DIR,
    selected_files=SELECTED_FILES,
    force=False,
)

print("\nSYNC CONCLUÍDO")
print(f"Sincronizados: {stats['synced']}")
print(f"Pulados: {stats['skipped']}")
print(f"Erros: {stats['errors']}")